# ASTE (Aspect Sentiment Triplet Extraction) - Dataset Analysis

This notebook loads and processes both **Restaurant (REST)** and **Laptop (LAP)** review datasets from the `nlp_assignment` folder.

Each record contains:
- **ID**: Unique identifier
- **Text**: The review sentence
- **Quadruplet**: List of `{Aspect, Opinion, Category, VA}` tuples

In [1]:
import json
import pandas as pd
from collections import Counter
from pathlib import Path

## 1. Load Both Datasets with Domain Keyword (REST / LAP)

In [2]:
DATA_DIR = Path("nlp_assignment")

def load_jsonl(filepath):
    """Load a JSONL file and return a list of dicts."""
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# Load raw records
rest_records = load_jsonl(DATA_DIR / "restaurant_train.jsonl")
lap_records  = load_jsonl(DATA_DIR / "laptop_train.jsonl")

print(f"Restaurant (REST) records: {len(rest_records)}")
print(f"Laptop     (LAP)  records: {len(lap_records)}")

Restaurant (REST) records: 2284
Laptop     (LAP)  records: 4076


## 2. Flatten Quadruplets into a DataFrame (with Domain Tag)

In [3]:
def flatten_records(records, domain_tag):
    """
    Flatten each record's Quadruplet list into individual rows.
    Adds a 'Domain' column (REST or LAP) for differentiation.
    """
    rows = []
    for rec in records:
        rid  = rec["ID"]
        text = rec["Text"]
        for quad in rec["Quadruplet"]:
            va = quad.get("VA", "")
            # VA is formatted as "valence#arousal"
            valence, arousal = (va.split("#") + ["", ""])[:2]
            rows.append({
                "Domain":   domain_tag,
                "ID":       rid,
                "Text":     text,
                "Aspect":   quad.get("Aspect", "NULL"),
                "Opinion":  quad.get("Opinion", "NULL"),
                "Category": quad.get("Category", ""),
                "Valence":  float(valence) if valence else None,
                "Arousal":  float(arousal) if arousal else None,
            })
    return pd.DataFrame(rows)

df_rest = flatten_records(rest_records, domain_tag="REST")
df_lap  = flatten_records(lap_records,  domain_tag="LAP")

# Combined dataframe
df = pd.concat([df_rest, df_lap], ignore_index=True)

print(f"REST quadruplets: {len(df_rest)}")
print(f"LAP  quadruplets: {len(df_lap)}")
print(f"Total rows:       {len(df)}")
print()
df.head(10)

REST quadruplets: 3659
LAP  quadruplets: 5773
Total rows:       9432



,Domain,ID,Text,Aspect,Opinion,Category,Valence,Arousal
0,REST,rest16_quad_dev_1,ca n ' t wait wait for my next visit .,NULL,NULL,RESTAURANT#GENERAL,6.75,6.38
1,REST,rest16_quad_dev_2,"their sake list was extensive , but we were lo...",sake list,extensive,DRINKS#STYLE_OPTIONS,7.83,8.00
2,REST,rest16_quad_dev_2,"their sake list was extensive , but we were lo...",NULL,NULL,SERVICE#GENERAL,5.00,5.00
3,REST,rest16_quad_dev_3,the spicy tuna roll was unusually good and the...,spicy tuna roll,unusually good,FOOD#QUALITY,7.50,7.62
4,REST,rest16_quad_dev_3,the spicy tuna roll was unusually good and the...,rock shrimp tempura,awesome,FOOD#QUALITY,8.25,8.38
5,REST,rest16_quad_dev_4,we love th pink pony .,pink pony,love,RESTAURANT#GENERAL,7.17,7.00
6,REST,rest16_quad_dev_5,this place has got to be the best japanese res...,place,best,RESTAURANT#GENERAL,7.88,8.12
7,REST,rest16_quad_dev_6,i tend to judge a sushi restaurant by its sea ...,sea urchin,heavenly,FOOD#QUALITY,7.70,7.60
8,REST,rest16_quad_dev_7,the prix fixe menu is worth every penny and yo...,prix fixe menu,worth,FOOD#PRICES,7.70,7.50
9,REST,rest16_quad_dev_7,the prix fixe menu is worth every penny and yo...,prix fixe menu,worth,FOOD#QUALITY,7.70,7.50


## 3. Quick Summary Statistics

In [4]:
print("=" * 60)
print("Dataset Statistics")
print("=" * 60)

for tag in ["REST", "LAP"]:
    sub = df[df["Domain"] == tag]
    n_sentences = sub["ID"].nunique()
    n_quads     = len(sub)
    n_categories = sub["Category"].nunique()
    null_aspect = (sub["Aspect"] == "NULL").sum()
    null_opinion = (sub["Opinion"] == "NULL").sum()
    avg_valence = sub["Valence"].mean()
    avg_arousal = sub["Arousal"].mean()

    print(f"\n--- {tag} ---")
    print(f"  Sentences:           {n_sentences}")
    print(f"  Quadruplets:         {n_quads}")
    print(f"  Avg quads/sentence:  {n_quads / n_sentences:.2f}")
    print(f"  Unique Categories:   {n_categories}")
    print(f"  NULL Aspects:        {null_aspect} ({100*null_aspect/n_quads:.1f}%)")
    print(f"  NULL Opinions:       {null_opinion} ({100*null_opinion/n_quads:.1f}%)")
    print(f"  Avg Valence:         {avg_valence:.2f}")
    print(f"  Avg Arousal:         {avg_arousal:.2f}")

Dataset Statistics

--- REST ---
  Sentences:           2284
  Quadruplets:         3659
  Avg quads/sentence:  1.60
  Unique Categories:   14
  NULL Aspects:        880 (24.1%)
  NULL Opinions:       699 (19.1%)
  Avg Valence:         6.22
  Avg Arousal:         6.84

--- LAP ---
  Sentences:           4076
  Quadruplets:         5773
  Avg quads/sentence:  1.42
  Unique Categories:   121
  NULL Aspects:        1254 (21.7%)
  NULL Opinions:       1583 (27.4%)
  Avg Valence:         5.94
  Avg Arousal:         6.67


## 4. Category Distribution (Both Domains)

In [5]:
print("\n=== TOP 15 CATEGORIES (REST) ===")
print(df_rest["Category"].value_counts().head(15).to_string())

print("\n=== TOP 15 CATEGORIES (LAP) ===")
print(df_lap["Category"].value_counts().head(15).to_string())


=== TOP 15 CATEGORIES (REST) ===
Category
FOOD#QUALITY                1282
SERVICE#GENERAL              680
RESTAURANT#GENERAL           579
AMBIENCE#GENERAL             398
FOOD#STYLE_OPTIONS           192
RESTAURANT#MISCELLANEOUS     135
FOOD#PRICES                  111
RESTAURANT#PRICES            101
DRINKS#QUALITY                66
DRINKS#STYLE_OPTIONS          47
LOCATION#GENERAL              42
DRINKS#PRICES                 24
FOOD#GENERAL                   1
SERVICE#QUALITY                1

=== TOP 15 CATEGORIES (LAP) ===
Category
LAPTOP#GENERAL                    1200
LAPTOP#OPERATION_PERFORMANCE       632
LAPTOP#DESIGN_FEATURES             489
LAPTOP#PRICE                       208
LAPTOP#QUALITY                     201
KEYBOARD#GENERAL                   156
DISPLAY#GENERAL                    145
BATTERY#OPERATION_PERFORMANCE      144
DISPLAY#OPERATION_PERFORMANCE      141
KEYBOARD#DESIGN_FEATURES           140
DISPLAY#DESIGN_FEATURES            128
LAPTOP#USABILITY        

## 5. Extract ASTE Triplets: (Aspect, Opinion, Sentiment)

We derive sentiment from the **Valence** score:
- Valence > 6  → **Positive**
- Valence < 4  → **Negative**
- Otherwise    → **Neutral**

In [6]:
def valence_to_sentiment(v):
    """Map valence score to a sentiment label."""
    if v is None:
        return "Neutral"
    if v > 6:
        return "Positive"
    elif v < 4:
        return "Negative"
    else:
        return "Neutral"

df["Sentiment"] = df["Valence"].apply(valence_to_sentiment)

print("Sentiment Distribution (overall):")
print(df["Sentiment"].value_counts().to_string())
print()

for tag in ["REST", "LAP"]:
    print(f"Sentiment Distribution ({tag}):")
    print(df[df["Domain"] == tag]["Sentiment"].value_counts().to_string())
    print()

Sentiment Distribution (overall):
Sentiment
Positive    5682
Neutral     2162
Negative    1588

Sentiment Distribution (REST):
Sentiment
Positive    2318
Neutral      795
Negative     546

Sentiment Distribution (LAP):
Sentiment
Positive    3364
Neutral     1367
Negative    1042



## 6. Build ASTE Triplet List per Sentence

In [7]:
def build_triplets(dataframe):
    """
    Group by sentence ID and collect (Aspect, Opinion, Sentiment) triplets.
    Returns a list of dicts: {Domain, ID, Text, Triplets}.
    """
    grouped = dataframe.groupby(["Domain", "ID", "Text"])
    results = []
    for (domain, sid, text), grp in grouped:
        triplets = []
        for _, row in grp.iterrows():
            triplets.append((
                row["Aspect"],
                row["Opinion"],
                row["Sentiment"]
            ))
        results.append({
            "Domain":   domain,
            "ID":       sid,
            "Text":     text,
            "Triplets": triplets,
        })
    return results

triplet_data = build_triplets(df)
print(f"Total sentences with triplets: {len(triplet_data)}")
print()

# Show a few examples from each domain
for tag in ["REST", "LAP"]:
    print(f"\n--- Sample {tag} ASTE Triplets ---")
    examples = [t for t in triplet_data if t["Domain"] == tag][:5]
    for ex in examples:
        print(f"  [{ex['Domain']}] {ex['ID']}")
        print(f"    Text: {ex['Text'][:100]}...")
        for a, o, s in ex["Triplets"]:
            print(f"      -> ({a}, {o}, {s})")
        print()

Total sentences with triplets: 6360


--- Sample REST ASTE Triplets ---
  [REST] rest16_quad_dev_1
    Text: ca n ' t wait wait for my next visit ....
      -> (NULL, NULL, Positive)

  [REST] rest16_quad_dev_10
    Text: the ambience is pretty and nice for conversation , so a casual lunch here would probably be best ....
      -> (ambience, pretty, Positive)
      -> (ambience, nice, Positive)
      -> (NULL, best, Positive)

  [REST] rest16_quad_dev_100
    Text: best drumsticks over rice and sour spicy soup in town !...
      -> (drumsticks over rice, best, Positive)
      -> (sour spicy soup, best, Positive)

  [REST] rest16_quad_dev_101
    Text: for those that go once and do n ' t enjoy it , all i can say is that they just do n ' t get it ....
      -> (NULL, NULL, Neutral)

  [REST] rest16_quad_dev_102
    Text: not worth it ....
      -> (NULL, not worth, Negative)


--- Sample LAP ASTE Triplets ---
  [LAP] laptop_quad_dev_1
    Text: this unit is ` ` pretty ` ` and stylish , s

## 7. Create a Unified Triplet DataFrame (for downstream use)

In [8]:
# Triplet-level DataFrame with Domain tag
df_triplets = df[["Domain", "ID", "Text", "Aspect", "Opinion", "Sentiment", "Category"]].copy()
df_triplets.head(10)

,Domain,ID,Text,Aspect,Opinion,Sentiment,Category
0,REST,rest16_quad_dev_1,ca n ' t wait wait for my next visit .,NULL,NULL,Positive,RESTAURANT#GENERAL
1,REST,rest16_quad_dev_2,"their sake list was extensive , but we were lo...",sake list,extensive,Positive,DRINKS#STYLE_OPTIONS
2,REST,rest16_quad_dev_2,"their sake list was extensive , but we were lo...",NULL,NULL,Neutral,SERVICE#GENERAL
3,REST,rest16_quad_dev_3,the spicy tuna roll was unusually good and the...,spicy tuna roll,unusually good,Positive,FOOD#QUALITY
4,REST,rest16_quad_dev_3,the spicy tuna roll was unusually good and the...,rock shrimp tempura,awesome,Positive,FOOD#QUALITY
5,REST,rest16_quad_dev_4,we love th pink pony .,pink pony,love,Positive,RESTAURANT#GENERAL
6,REST,rest16_quad_dev_5,this place has got to be the best japanese res...,place,best,Positive,RESTAURANT#GENERAL
7,REST,rest16_quad_dev_6,i tend to judge a sushi restaurant by its sea ...,sea urchin,heavenly,Positive,FOOD#QUALITY
8,REST,rest16_quad_dev_7,the prix fixe menu is worth every penny and yo...,prix fixe menu,worth,Positive,FOOD#PRICES
9,REST,rest16_quad_dev_7,the prix fixe menu is worth every penny and yo...,prix fixe menu,worth,Positive,FOOD#QUALITY


In [9]:
# Filter by domain easily
df_rest_triplets = df_triplets[df_triplets["Domain"] == "REST"]
df_lap_triplets  = df_triplets[df_triplets["Domain"] == "LAP"]

print(f"REST triplets: {len(df_rest_triplets)}")
print(f"LAP  triplets: {len(df_lap_triplets)}")

REST triplets: 3659
LAP  triplets: 5773


## 8. Top Aspects and Opinions per Domain

In [10]:
for tag in ["REST", "LAP"]:
    sub = df_triplets[df_triplets["Domain"] == tag]
    non_null_aspects  = sub[sub["Aspect"]  != "NULL"]["Aspect"]
    non_null_opinions = sub[sub["Opinion"] != "NULL"]["Opinion"]

    print(f"\n=== TOP 10 ASPECTS ({tag}) ===")
    print(non_null_aspects.value_counts().head(10).to_string())

    print(f"\n=== TOP 10 OPINIONS ({tag}) ===")
    print(non_null_opinions.value_counts().head(10).to_string())


=== TOP 10 ASPECTS (REST) ===
Aspect
food          326
service       231
place         177
restaurant     71
staff          56
sushi          55
pizza          44
atmosphere     43
decor          39
menu           26

=== TOP 10 OPINIONS (REST) ===
Opinion
great        225
good         138
best          91
delicious     69
excellent     61
nice          46
love          41
amazing       40
very good     32
friendly      29

=== TOP 10 ASPECTS (LAP) ===
Aspect
laptop          416
screen          241
keyboard        219
chromebook      218
computer        216
battery life    145
product         106
machine          98
battery          94
device           72

=== TOP 10 OPINIONS (LAP) ===
Opinion
great        321
good         169
love         151
nice         117
fast         103
well          73
excellent     72
easy          59
perfect       58
amazing       53


## 9. Sentiment-Category Cross-Tabulation

In [11]:
for tag in ["REST", "LAP"]:
    sub = df_triplets[df_triplets["Domain"] == tag]
    ct = pd.crosstab(sub["Category"], sub["Sentiment"])
    ct["Total"] = ct.sum(axis=1)
    ct = ct.sort_values("Total", ascending=False)
    print(f"\n=== Sentiment x Category ({tag}) ===")
    print(ct.head(15).to_string())
    print()


=== Sentiment x Category (REST) ===
Sentiment                 Negative  Neutral  Positive  Total
Category                                                    
FOOD#QUALITY                   175      211       896   1282
SERVICE#GENERAL                168      185       327    680
RESTAURANT#GENERAL              75      138       366    579
AMBIENCE#GENERAL                32       51       315    398
FOOD#STYLE_OPTIONS              19       55       118    192
RESTAURANT#MISCELLANEOUS        11       57        67    135
FOOD#PRICES                     34       32        45    111
RESTAURANT#PRICES               23       38        40    101
DRINKS#QUALITY                   1        9        56     66
DRINKS#STYLE_OPTIONS             2        2        43     47
LOCATION#GENERAL                 2       10        30     42
DRINKS#PRICES                    3        6        15     24
FOOD#GENERAL                     0        1         0      1
SERVICE#QUALITY                  1        0     

## 10. Summary

| Property | REST | LAP |
|---|---|---|
| Domain Keyword | `REST` | `LAP` |
| Source File | `restaurant_train.jsonl` | `laptop_train.jsonl` |
| Fields per Entry | ID, Text, Quadruplet (Aspect, Opinion, Category, VA) | ID, Text, Quadruplet (Aspect, Opinion, Category, VA) |

Both datasets are now loaded, tagged, and available as:
- `df` — combined flat DataFrame
- `df_triplets` — with Sentiment column
- `triplet_data` — per-sentence ASTE triplet list
- Filter by `Domain == "REST"` or `Domain == "LAP"`